# Carga, variables y calidad de los datos

## Entorno

Importé las librerías y fijé la semilla 42, que usan la partición y todos los modelos. Sin semilla,
K-Means y GMM arrancan desde centroides o componentes distintos en cada corrida, así que los grupos, su
numeración y las métricas cambiarían, y nadie podría verificar las cifras del informe. También imprimí
las versiones de Python, NumPy, pandas, scikit-learn y Matplotlib, porque los valores por defecto de los
estimadores cambian de una versión a otra y con ellos los resultados.

Definí tres funciones auxiliares. La función redondear aplica redondeo mitad hacia arriba una sola vez
sobre el valor completo. La usé en todas las cifras que se guardan porque round de Python trabaja sobre
la representación binaria y, por ejemplo, redondea 0,2175 a 0,217, mientras que el informe diría 0,218.
Así cada número se ve igual en el notebook y en el informe. La función guardar escribe la figura en
splits antes de mostrarla, porque plt.show libera la figura y el archivo saldría en blanco. La función
guardar_resultados agrega las cifras de cada notebook a resultados.json, de donde las toma el informe sin
copiarlas a mano.

In [1]:
import os, time, json, platform, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import sklearn
import joblib
from decimal import Decimal, ROUND_HALF_UP

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.dpi'] = 100
SEMILLA = 42
np.random.seed(SEMILLA)

EN_DRIVE = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.isdir('/content/drive/MyDrive'):
        raise RuntimeError('Drive no quedó montado. Vuelve a ejecutar esta celda y autoriza el acceso '
                           'en la ventana emergente.')
    RUTA = '/content/drive/MyDrive/credit_card'
    EN_DRIVE = True
except ImportError:
    RUTA = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')

CARPETA_SPLITS = os.path.join(RUTA, 'splits')
os.makedirs(CARPETA_SPLITS, exist_ok=True)
ARCHIVO_RESULTADOS = os.path.join(RUTA, 'resultados.json')
resultados = {}
if os.path.exists(ARCHIVO_RESULTADOS):
    with open(ARCHIVO_RESULTADOS, encoding='utf-8') as f:
        resultados = json.load(f)

def redondear(valor, decimales=3):
    r = Decimal(repr(float(valor))).quantize(Decimal(1).scaleb(-decimales), rounding=ROUND_HALF_UP)
    return int(r) if decimales == 0 else float(r)

def redondear_serie(serie, decimales=3):
    return serie.map(lambda v: redondear(v, decimales))

def guardar(nombre):
    plt.tight_layout()
    plt.savefig(os.path.join(CARPETA_SPLITS, nombre + '.png'), dpi=150, bbox_inches='tight', facecolor='white')
    print('Gráfico guardado:', nombre + '.png')
    plt.show()

def guardar_resultados():
    with open(ARCHIVO_RESULTADOS, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=1,
                  default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    print('Cifras guardadas en resultados.json')

resultados['versiones'] = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                           'scikit-learn': sklearn.__version__, 'matplotlib': matplotlib.__version__}
print('Guardando en Google Drive' if EN_DRIVE else 'Google Drive no disponible, guardando localmente')
print('Versiones:', resultados['versiones'])

Google Drive no disponible, guardando localmente
Versiones: {'python': '3.12.10', 'numpy': '2.2.6', 'pandas': '3.0.3', 'scikit-learn': '1.8.0', 'matplotlib': '3.10.9'}


## Carga del conjunto de datos

Usé el Credit Card Dataset for Clustering, que Arjun Bhasin publicó en Kaggle con licencia CC0, de
dominio público. Cada fila resume seis meses de uso de la tarjeta de un cliente activo. Lo elegí porque
describe la conducta de uso, que es lo que un banco mide de cada cliente, y porque no trae
ninguna columna con el tipo de cliente. Sin esa etiqueta el problema es de agrupamiento y no de
clasificación, y los grupos tienen que salir del propio comportamiento de los clientes. Leí la copia
local CC_GENERAL.csv si ya existía y, si no, descargué el conjunto arjunbhasin2013/ccdata de Kaggle con
kagglehub, que no pide credenciales para datos públicos. Guardar la copia evita descargarlo en cada
notebook y asegura que los seis trabajen sobre el mismo archivo.

In [2]:
warnings.filterwarnings('ignore', message='IProgress not found')
import kagglehub

ARCHIVO_LOCAL = os.path.join(RUTA, 'CC_GENERAL.csv')

def cargar_datos():
    if os.path.exists(ARCHIVO_LOCAL):
        return pd.read_csv(ARCHIVO_LOCAL)
    carpeta = kagglehub.dataset_download('arjunbhasin2013/ccdata')
    datos = pd.read_csv(os.path.join(carpeta, 'CC GENERAL.csv'))
    datos.to_csv(ARCHIVO_LOCAL, index=False)
    return datos

datos = cargar_datos()
print('Filas:', datos.shape[0], '| Columnas:', datos.shape[1])

Filas: 8950 | Columnas: 18


In [3]:
datos.head()

,CUST_ID,BALANCE,BALANCE_FREQUENCY,PURCHASES,ONEOFF_PURCHASES,INSTALLMENTS_PURCHASES,CASH_ADVANCE,PURCHASES_FREQUENCY,ONEOFF_PURCHASES_FREQUENCY,PURCHASES_INSTALLMENTS_FREQUENCY,CASH_ADVANCE_FREQUENCY,CASH_ADVANCE_TRX,PURCHASES_TRX,CREDIT_LIMIT,PAYMENTS,MINIMUM_PAYMENTS,PRC_FULL_PAYMENT,TENURE
0,C10001,40.900749,0.818182,95.40,0.00,95.4,0.000000,0.166667,0.000000,0.083333,0.000000,0,2,1000.0,201.802084,139.509787,0.000000,12
1,C10002,3202.467416,0.909091,0.00,0.00,0.0,6442.945483,0.000000,0.000000,0.000000,0.250000,4,0,7000.0,4103.032597,1072.340217,0.222222,12
2,C10003,2495.148862,1.000000,773.17,773.17,0.0,0.000000,1.000000,1.000000,0.000000,0.000000,0,12,7500.0,622.066742,627.284787,0.000000,12
3,C10004,1666.670542,0.636364,1499.00,1499.00,0.0,205.788017,0.083333,0.083333,0.000000,0.083333,1,1,7500.0,0.000000,NaN,0.000000,12
4,C10005,817.714335,1.000000,16.00,16.00,0.0,0.000000,0.083333,0.083333,0.000000,0.000000,0,1,1200.0,678.334763,244.791237,0.000000,12


## Descripción de variables

CUST_ID identifica al cliente y no entra a ningún modelo. Las otras 17 columnas son numéricas y miden
cuatro aspectos del uso de la tarjeta.

| Dimensión | Variables | Qué miden |
|---|---|---|
| Gasto | PURCHASES, ONEOFF_PURCHASES, INSTALLMENTS_PURCHASES, PURCHASES_TRX | Monto comprado en total, en un solo pago y en cuotas, y número de compras |
| Frecuencia | PURCHASES_FREQUENCY, ONEOFF_PURCHASES_FREQUENCY, PURCHASES_INSTALLMENTS_FREQUENCY | Qué tan seguido compra, de 0 a 1 |
| Efectivo | CASH_ADVANCE, CASH_ADVANCE_FREQUENCY, CASH_ADVANCE_TRX | Monto, frecuencia y número de avances en efectivo |
| Pago y deuda | BALANCE, BALANCE_FREQUENCY, PAYMENTS, MINIMUM_PAYMENTS, PRC_FULL_PAYMENT, CREDIT_LIMIT, TENURE | Saldo adeudado, pagos, pago mínimo, proporción del pago total, límite y antigüedad en meses |

Conté los tipos de dato y comprobé que el conjunto tuviera al menos 500 observaciones y 8 variables
numéricas, el mínimo que pide la consigna para que reducir la dimensionalidad tenga sentido. También
confirmé que no hubiera variables categóricas, porque K-Means y GMM solo trabajan con distancias y
densidades sobre números.

In [4]:
numericas = datos.select_dtypes('number').columns.tolist()
print('=== Tipos de dato ===')
print(datos.dtypes.astype(str).value_counts().to_string(), '\n')
print('Variables numéricas:', len(numericas))
print('Cumple 500 observaciones y 8 variables numéricas:', len(datos) >= 500 and len(numericas) >= 8)
resultados['n_filas'], resultados['n_numericas'] = len(datos), len(numericas)

=== Tipos de dato ===
float64    14
int64       3
str         1 

Variables numéricas: 17
Cumple 500 observaciones y 8 variables numéricas: True


El conjunto tiene 8 950 clientes y 18 columnas, el identificador de texto y 17 variables numéricas, de
las cuales 14 son decimales y 3 son conteos enteros (PURCHASES_TRX, CASH_ADVANCE_TRX y TENURE). Supera
con holgura ese mínimo y no hay variables categóricas que codificar.

## Calidad de los datos

K-Means y GMM no aceptan valores faltantes, y una fila repetida pesaría doble al calcular los grupos. Por
eso conté los nulos por variable, las filas duplicadas y los identificadores repetidos. También busqué
proporciones mayores que 1 en las frecuencias y en PRC_FULL_PAYMENT, que por definición van de 0 a 1.

In [5]:
nulos = datos.isna().sum()
print('=== Nulos por variable ===')
print(nulos[nulos > 0].to_string(), '\n')
print('Filas duplicadas:', int(datos.duplicated().sum()))
print('Identificadores repetidos:', int(datos['CUST_ID'].duplicated().sum()))
frecuencias = [c for c in numericas if 'FREQUENCY' in c or c == 'PRC_FULL_PAYMENT']
fuera_rango = {c: int((datos[c] > 1).sum()) for c in frecuencias if (datos[c] > 1).any()}
print('Proporciones mayores que 1:', fuera_rango)
resultados['nulos'] = {k: int(v) for k, v in nulos[nulos > 0].items()}
resultados['fuera_rango'] = fuera_rango

=== Nulos por variable ===
CREDIT_LIMIT          1
MINIMUM_PAYMENTS    313 

Filas duplicadas: 0
Identificadores repetidos: 0
Proporciones mayores que 1: {'CASH_ADVANCE_FREQUENCY': 8}


Solo dos variables tienen nulos. MINIMUM_PAYMENTS tiene 313 (3,5 por ciento de los clientes) y
CREDIT_LIMIT tiene uno. Son pocos para descartar filas, así que los imputé con la mediana dentro del
pipeline del notebook 03, ajustada solo con entrenamiento. No hay filas ni identificadores repetidos.
CASH_ADVANCE_FREQUENCY supera 1 en 8 clientes, un error de registro porque una frecuencia no puede pasar
del 100 por ciento, y en el notebook 03 la recorté a 1.

Revisé la descriptiva de las 17 variables para ver en qué escala está cada una y si la media se aleja de
la mediana, que es la primera señal de una cola larga. Las dos cosas deciden cómo hay que transformar y
escalar antes de medir distancias.

In [6]:
datos[numericas].describe().T.apply(lambda c: redondear_serie(c, 2))

,count,mean,std,min,25%,50%,75%,max
BALANCE,8950.0,1564.47,2081.53,0.00,128.28,873.39,2054.14,19043.14
BALANCE_FREQUENCY,8950.0,0.88,0.24,0.00,0.89,1.00,1.00,1.00
PURCHASES,8950.0,1003.20,2136.63,0.00,39.64,361.28,1110.13,49039.57
ONEOFF_PURCHASES,8950.0,592.44,1659.89,0.00,0.00,38.00,577.41,40761.25
INSTALLMENTS_PURCHASES,8950.0,411.07,904.34,0.00,0.00,89.00,468.64,22500.00
CASH_ADVANCE,8950.0,978.87,2097.16,0.00,0.00,0.00,1113.82,47137.21
PURCHASES_FREQUENCY,8950.0,0.49,0.40,0.00,0.08,0.50,0.92,1.00
ONEOFF_PURCHASES_FREQUENCY,8950.0,0.20,0.30,0.00,0.00,0.08,0.30,1.00
PURCHASES_INSTALLMENTS_FREQUENCY,8950.0,0.36,0.40,0.00,0.00,0.17,0.75,1.00
CASH_ADVANCE_FREQUENCY,8950.0,0.14,0.20,0.00,0.00,0.00,0.22,1.50


Las escalas son muy distintas entre variables. Los montos y el límite de crédito se miden en miles, con
máximos de varias decenas de miles, mientras que las frecuencias y la proporción de pagos totales van de
0 a 1. En los montos la media supera con holgura a la mediana, señal de colas largas a la derecha. Ambas
cosas obligan a transformar y escalar antes de medir distancias, y el notebook 02 las cuantifica.

## Cierre del notebook

Guardé las cifras de este notebook en resultados.json.

In [7]:
guardar_resultados()

Cifras guardadas en resultados.json
